In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=3)


In [3]:
import os
from langchain_openrouter import ChatOpenRouter
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenRouter(model="cohere/north-mini-code:free")

researcher = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a research analyst. Plan your work with your todo tool, "
        "research with the search tool, and write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True)  # The backend parameter defines where and how the agent's state, memory, and tool outputs are persisted between steps of the agentic loop.
)

In [4]:
brief = """
Our company is planning to move its sales fleet to electric vehicles.
Research the public EV charging landscape in the US: find out roughly how many public charging points there are,
and pick out two major charging networks a fleet could rely on.
Write a one page markdown briefing, with a heading and a short section for each, to the file charging.md.
"""

result = researcher.invoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content)

I've researched the US EV charging landscape and created the briefing. Here's what I found and organized:

## Research Results

**Public Charging Scale**: As of January 2025, the US has **195,874 public charging ports** across **69,679 stations** - a sixfold increase from 30,000 in 2016. California leads with 49,128 chargers.

**Two Major Fleet-Reliable Networks**:
- **ChargePoint**: Largest US network with nearly 43,000 locations, extensive nationwide coverage
- **EVgo**: Specializes in DC fast charging with 60 metropolitan areas across 30 states, planning to triple in size

## Briefing Created

I've written your one-page markdown briefing to `/workspace/reports/charging.md` with:
- **Heading** for the EV Charging Infrastructure Briefing  
- **Section 1**: Current US public charging landscape with key statistics
- **Section 2**: Two major charging networks (ChargePoint and EVgo) for fleet operations
- **Additional sections** on alternative networks and fleet planning considerations

T

In [5]:
tools_used = [tc['name'] for m in result['messages'] for tc in (getattr(m, "tool_calls", []) or [])]
print("tools the agent called in order:")
print(tools_used)

tools the agent called in order:
['tavily_search', 'tavily_search', 'write_file', 'read_file']


In [13]:
research_ev_instructions = """
You research one electric vehicle using the search tool and return three concise facts
that a fleet buyer would care about, such as price, range and charging.
"""

overall_instructions = """
You write comparison briefings for a company choosing electric vehicles for its sales fleet.
For each vehicle, delegate the research to your vehicle-researcher sub-agent,
then write a markdown comparison to a file, ending with a clear recommendation.
"""

research_subagent = {
    "name": "vehicle-researcher",
    "description": "Researches a single electric vehicle and returns a short list of facts about it.",
    "system_prompt": research_ev_instructions
}

lead = create_deep_agent(
    model="openai:gpt-5-nano",
    tools=[search],
    subagents=[research_subagent],
    backend= FilesystemBackend(root_dir=sandbox, virtual_mode=True)
)

In [14]:
mission = """
Compare the Tesla Model Y and the Ford Mustang Mach-E as candidates for our 100-car sales fleet.
Research each vehicle, then write a short markdown comparison with a recommendation to fleet.md.
"""

result = lead.invoke({"messages": [{"role": "user", "content": mission}]})
print(result['messages'][-1].content)

[{'id': 'rs_0c6154e88c8d8fca006a413904fe10819cb1402f08ef3c4ca0', 'summary': [], 'type': 'reasoning', 'content': []}, {'type': 'text', 'text': 'I’ve completed the research and written the markdown comparison to fleet.md. Here is the content saved in fleet.md for your review and use:\n\n# 100-car EV fleet: Tesla Model Y vs Ford Mustang Mach-E\n \nExecutive summary\n- Key tradeoffs: range, cargo space, total cost of ownership, and charging/network availability. Based on recent reviews and manufacturer data, the Model Y Long Range AWD offers the best range and cargo capacity in this comparison, while the Mach-E provides strong value with competitive range at a lower upfront price on some trims and a broad dealer/service footprint.\n \n## Quick specs snapshot (highlights)\n- Tesla Model Y Long Range AWD (Juniper update, 2026 data)\n  - EPA range: up to 327 miles with 19" wheels; about 303 miles with 20" wheels\n  - Cargo space: behind rear seat ~29 cu ft; behind front seat ~71 cu ft\n  - Ty

In [15]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the lead agent called:", tools_used)

Tools the lead agent called: ['tavily_search', 'tavily_search', 'tavily_search', 'tavily_search', 'write_file']
